# Cloud Desk A/B Testing Pricing Page
**CloudDesk is** a B2B Saa company selling project management software to small and medium businesses. Customers sign up via a self-serve flow on the marketing site.

**Current state:** 3 plans 

- Starter £12,
- Pro £24,
- Business £48 per user/month.

Headline prices are monthly, with a small "or pay annually for 20% off" toggle at the bottom.

 Only ~18% of new signups choose annual.

**The team's hypothesis:** Annual customers are dramatically more valuable (better retention, predictable revenue), so push more users to annual by **redesigning the pricing page to default to annual pricing**  "$9.60/user/month, billed annually" with a "Save 20%" badge and demoting monthly to a secondary toggle.

The stated test hypothesis: *Shifting the default to annual will lift annual signup share, without significantly hurting overall signup conversion.*


In [41]:
import pandas as pd
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize ,proportions_ztest, proportion_confint 
from scipy.stats import chisquare ,ttest_ind

In [3]:
#Loading pre- period data 
pre = pd.read_csv('pre_period_visits.csv')
pre['visit_date'] = pd.to_datetime(pre['visit_date'])

print(f"Rows: {len(pre):,}")
print(f"Date range: {pre['visit_date'].min().date()} to {pre['visit_date'].max().date()}")
print(f"\nFirst 5 rows:")
print(pre.head())
print(f"\nColumn types:")
print(pre.dtypes)

Rows: 234,342
Date range: 2024-08-05 to 2024-09-01

First 5 rows:
    visit_id visit_date company_size country traffic_source  signed_up plan  \
0  V00000001 2024-08-05       50-199      UK         direct      False  NaN   
1  V00000002 2024-08-05        20-49      UK        organic      False  NaN   
2  V00000003 2024-08-05         5-19      CA         direct      False  NaN   
3  V00000004 2024-08-05          1-4      US       referral      False  NaN   
4  V00000005 2024-08-05         5-19   Other        organic      False  NaN   

  billing_cycle  first_year_revenue  
0           NaN                 0.0  
1           NaN                 0.0  
2           NaN                 0.0  
3           NaN                 0.0  
4           NaN                 0.0  

Column types:
visit_id                      object
visit_date            datetime64[ns]
company_size                  object
country                       object
traffic_source                object
signed_up                      

# Calculating the Baseline Metrics

In [11]:
# Baseline 1: overall signup conversion (visitors who signed up)

baseline_signup_rate = pre['signed_up'].mean()
print(f"Baseline signup conversion rate: {baseline_signup_rate:.2%}")

# Baseline 2: among signups, what % chose annual?
signups_only =pre[pre['signed_up']]
baseline_annual_share = (signups_only['billing_cycle'] == 'annual').mean()
print(f"Baseline annual share (of signups): {baseline_annual_share:.2%}")

# Baseline 3: revenue per visitor
baseline_rev_per_visitor = pre['first_year_revenue'].mean()
print(f"Baseline revenue per visitor: ${baseline_rev_per_visitor:.2f}")

# Baseline plan tier mix (among signups)
baseline_tier_mix = signups_only['plan'].value_counts(normalize=True).round(4)
print("Baseline plan tier mix (among signups):")
print(baseline_tier_mix)

daily_visitors = pre.groupby(pre['visit_date']).size()
print(f"\nDaily visitor stats:")
print(f"  Mean: {daily_visitors.mean():.0f}")
print(f"  Median: {daily_visitors.median():.0f}")
print(f"  Min: {daily_visitors.min()}")
print(f"  Max: {daily_visitors.max()}")

Baseline signup conversion rate: 4.73%
Baseline annual share (of signups): 19.81%
Baseline revenue per visitor: $8.09
Baseline plan tier mix (among signups):
pro         0.4153
starter     0.3540
business    0.2307
Name: plan, dtype: float64

Daily visitor stats:
  Mean: 8369
  Median: 9438
  Min: 4342
  Max: 10679


# Observation1  
### Baseline Signup Conversion Rate is around 5% - Guadrail Metric Baseline
### Baseline Annual Share is around 20%  - Primary Metric
### Baseline Revenue per visitor  is around £8  - 2nd Guadrail metric Baseline

# Defining the Primary Metric
The teams goal is to increase the share of new signups choosing annual. 

**Annual share of new signups** — defined as the proportion of users who completed
a signup and chose the annual billing cycle, out of all completed signups.

Primary Metric : Annual share of new signups = signups choosing annual / total signups   

Annual customers retain better and have predictable revenue, so increasing this share directly reflects the intended
strategic outcome.

**Baseline (from pre-period data):** 19.8% of signups choose annual.

# Defining the Guadrail Metrics  

The goal is to lift annual share without hurting overall signup conversion .   

### Guardrail 1: Overall signup conversion rate  

**Definition:** `total_signups / total_visitors`  

**Why:** the most likely con of this redesign is that visitors see the higher annual headline price ("£9.60/user/month, billed annually" implies £115 upfront for one user) and bounce.
If that happens, we'd see fewer signups overall — even as the share who do sign up choosing annual goes up. 

**Baseline:** 4.7% (from pre-period).


**Decision threshold:** I would be uncomfortable shipping if signup conversion drops by more than 0.3 percentage points (i.e. below ~4.4%) with statistical significance.

### Guardrail 2: First-year revenue per visitor

**Definition:** `total_first_year_revenue / total_visitors`

**Why:** this is the metric the business actually cares about. The trade-off between "fewer signups but more annual" vs "more signups but more monthly" is ambiguous in isolation — revenue per visitor should solve it.

**Baseline:** £8.09 per visitor.

**Decision threshold:** revenue per visitor must not drop materially. 
A flat or positive movement supports shipping; a clear negative movement is a kill signal
even if annual share lifted.



### Guardrail 3: Plan tier mix (Starter / Pro / Business)

**Definition:** % of signups choosing each tier, by variant.

**Why:** a more subtle risk. The annual-default treatment shows the "£9.60/user
/month" annual price for Starter prominently, but the Pro annual price (£19.20)
and Business annual price (£38.40) are still substantial. There's a chance the
redesign makes Pro and Business look more expensive by comparison, pushing
signups toward Starter. A tier downshift would hurt revenue independently of
signup volume.

**Baseline (from pre-period data):**
- Starter: 35.4%
- Pro: 41.5%
- Business: 23.1%

**Decision threshold:**  I would want to investigate but probably not block on this guardrail alone.

# Sample Size and Power Calculation

How many signups do I need before the test can give me a reliable answer?  

- **Baseline annual share:** 19.81% (from pre-period data)
- **Minimum detectable effect (MDE):** +5 percentage points. A smaller lift
  wouldn't justify the engineering and design cost of the redesign; a larger
  MDE risks designing the test to miss real-but-modest wins.
- **Alpha (α):** 0.05  industry-standard false positive rate.
- **Power (1−β):** 0.80  industry-standard true positive rate.

I am using a two-proportion test (since the primary metric is a proportion:
share of signups choosing annual).

In [14]:
baseline_annual_share = 0.1981
mde_pp = 0.05
treatment_target = baseline_annual_share + mde_pp
alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(treatment_target, baseline_annual_share)
analysis = NormalIndPower()
n_per_arm = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

print(f"Required sample size per arm: {int(np.ceil(n_per_arm)):,} signups")

Required sample size per arm: 1,086 signups


1,086 signups per arm is the minimum you need before the test can reliably detect a 5pp lift in annual share.  
Since this is run on visitors, we need to convert this into visitor counts then to test duration .

1. How many vistors per arm do i need?
2. How long will it take to get there at our current traffic?

In [17]:
# Calculate Visitors needed per arm 
required_signups_per_arm = 1086
baseline_signup_rate = 0.0473  # from pre-period

visitors_per_arm = required_signups_per_arm / baseline_signup_rate
total_visitors_needed = visitors_per_arm * 2  # control + treatment

print(f"Required signups per arm: {required_signups_per_arm:,}")
print(f"Required visitors per arm: {int(np.ceil(visitors_per_arm)):,}")
print(f"Total visitors needed (both arms): {int(np.ceil(total_visitors_needed)):,}")

Required signups per arm: 1,086
Required visitors per arm: 22,960
Total visitors needed (both arms): 45,920


In [18]:
#Calculate Test duration  
#Daily visitor mean is ~8,400. 
#Split 50/50 between control and treatment, that's ~4,200 visitors per arm per day:

daily_visitors = 8369  # mean from  earlier output
visitors_per_arm_per_day = daily_visitors / 2

days_needed = visitors_per_arm / visitors_per_arm_per_day
print(f"Visitors per arm per day: {visitors_per_arm_per_day:,.0f}")
print(f"Days needed to reach required sample: {days_needed:.1f}")

Visitors per arm per day: 4,184
Days needed to reach required sample: 5.5


At current traffic of  around 8,400 visitors per day split 50/50, each arm receives  around 4,200 visitors per day. 
The minimum sample size would therefore be reached in **~5.5
days**. 

However, **5 days is not the right test duration.** I'm planning for 4 full weeks (Sep 2 - Sep 29, 2024) for three reasons:  
1. **Weekly cycles.** B2B SaaS traffic varies sharply by day of week (pre-period
   range: 4,342 to 10,679 daily visitors). Running for less than a full week
   over-samples weekday visitors, who may behave differently to weekend ones.
   The minimum sensible duration is 2 weeks; 4 weeks captures multiple cycles
   for stability.

2. **Novelty effects.** Users seeing a redesigned page may click on it out of
   curiosity rather than genuine preference. This bias fades over 1-2 weeks.
   Running 4 weeks lets us explicitly test whether the effect sustains beyond
   the novelty period.

3. **Subgroup statistical power.** The pre-registered subgroup analyses (by
   company size, by traffic source) operate on smaller slices of the data —
   roughly 1/5 of the sample for each company size cut. The headline test
   minimum doesn't give us meaningful power within subgroups; a longer test
   does.  
   
   **Expected sample size at 4 weeks:** ~235,000 visitors total, about 5x the
minimum. This buffer is appropriate because (a) we want to detect subgroup
effects, and (b) the treatment may reduce signup conversion (the guardrail),
which would slow signup accumulation in that arm.

# Randomisation and Assignment 

At what level do you randomise? Per-visit? Per-session? Per-user?
How is the assignment actually done? What ensures it's truly random and 50/50?

**Randomisation unit:** visitor-level.  
I am choosing visitor level since that user will always see that variant on every future visit . It can be implemented via cookie or hashed visitor ID .

**Why visitor-level:**  the pricing page is something a visitor might see multiple times before signing up  they research, leave, come back, decide. If we showed them different variants on different visits, their final decision would be informed by both versions, blurring the comparison.  

**Assignment method:** deterministic hash-based assignment.  
Each new visitor is assigned a stable identifier (a hashed visitor ID stored
client-side). Their variant is then computed by hashing this ID and taking
the result modulo 2 — even hash would be control, odd hash  would be treatment.
This produces
a 50/50 split.


**Split:** 50/50 between control and treatment.

A 50/50 split maximises statistical power per visitor exposed. An imbalanced
split (e.g. 90/10) is sometimes used when treatment is risky, but here the
downside is bounded , worst case, signup conversion drops by a few percentage
points in treatment for four weeks, recoverable on rollback.

## Pre-registered subgroup analyses    


### 1. By company size

The treatment changes the headline price visibility. Smaller companies are
typically more price-sensitive, so any negative effect on signup conversion
will likely concentrate in the 1-4 and 5-19 company sizes. Conversely, the
annual lift may be larger for bigger companies who are already accustomed
to annual contracts.

**Hypothesis:** treatment lifts annual share more for larger companies, and
hurts signup conversion more for smaller companies.

### 2. By traffic source

Different sources reflect different intent levels. Organic, direct, and
referral traffic is generally higher-intent and less likely to bounce on
price changes. Paid traffic (paid_social, paid_search) is typically lower-
intent and more sensitive to friction.

**Hypothesis:** higher-intent traffic shows smaller (or no) negative impact
on signup conversion. Lower-intent traffic shows a larger drop.

### 3. By country

Annual subscription preference and price sensitivity vary regionally. This
is more of a "no surprises" check than a strong directional hypothesis
mainly to ensure the treatment doesn't have wildly inconsistent effects
across markets.

**Hypothesis:** no strong prior. Flagging if any country shows directionally
opposite effects to the global average.

### What I am NOT pre-registering

Any cuts by plan tier, day-of-week, time-of-day, or anything not listed above.
If I find an interesting effect in those during analysis, I will report it as
exploratory and propose a follow-up test rather than treating it as confirmed.

## Decision criteria (committed in advance)

I commit to the following decision rules before seeing the test data. If
the test produces an outcome that doesn't cleanly fit one of these rules,
I'll fall back on the most conservative interpretation (typically: don't
ship, run a follow-up).

### Ship if:

- Primary metric (annual share) lifts by ≥3 percentage points with p<0.05
- AND signup conversion does not drop by more than 0.3pp (within statistical
  noise)
- AND first-year revenue per visitor is flat or positive
- AND no pre-registered subgroup shows a directionally opposite effect

### Kill if:

- Primary metric does not lift, or lifts by less than 3pp with no statistical
  significance
- OR signup conversion drops by more than 1pp with p<0.05 (regardless of
  primary metric movement)
- OR revenue per visitor drops materially (>5% relative drop)

### Iterate if:

- Primary metric lifts meaningfully (≥3pp) AND a guardrail breaks
  significantly (e.g. signup conversion drops 0.3-1pp). The signal is real
  but the cost is too high - the version we tested isn't the right one.
- Subgroup analyses suggest the effect is highly segment-specific (e.g.
  treatment works for enterprise but hurts SMB). A segmented rollout or
  redesign for the affected segment may resolve it.

### Run a follow-up if:

- Results are directionally promising but underpowered (e.g. primary moves
  in the right direction but doesn't reach significance)
- Novelty effects are visible (early weeks differ markedly from late weeks)
  and a longer test would clarify
- An unexpected, unplanned effect needs validation (no acting on
  exploratory findings without a confirmatory test)

### A note on the MDE threshold (3pp)

The power calculation was sized for a 5pp MDE, but I'm using 3pp as the
shipping threshold. Why? Because 5pp was the *minimum effect I designed the
test to detect* - meaning at 5pp I'd have 80% power to find statistical
significance. A 3pp lift will likely also reach significance with the
sample buffer I have (4 weeks vs minimum 5 days). 3pp is also still a
practically meaningful business outcome.

# SECTION 2: TEST Analysis

In [20]:
# Now we load the test period data
test = pd.read_csv('test_period_visits.csv')
test['visit_date'] = pd.to_datetime(test['visit_date'])

print(f"Test rows: {len(test):,}")
print(f"Date range: {test['visit_date'].min().date()} to {test['visit_date'].max().date()}")
print(f"\nFirst 5 rows:")
print(test.head())
print(f"\nColumns: {list(test.columns)}")

Test rows: 241,127
Date range: 2024-09-02 to 2024-09-29

First 5 rows:
    visit_id visit_date    variant company_size country traffic_source  \
0  V00234343 2024-09-02  treatment       50-199      US        organic   
1  V00234344 2024-09-02    control          1-4      US         direct   
2  V00234345 2024-09-02    control         5-19      UK         direct   
3  V00234346 2024-09-02    control          1-4      US         direct   
4  V00234347 2024-09-02    control         5-19      CA    paid_search   

   signed_up plan billing_cycle  first_year_revenue  
0      False  NaN           NaN                 0.0  
1      False  NaN           NaN                 0.0  
2      False  NaN           NaN                 0.0  
3      False  NaN           NaN                 0.0  
4      False  NaN           NaN                 0.0  

Columns: ['visit_id', 'visit_date', 'variant', 'company_size', 'country', 'traffic_source', 'signed_up', 'plan', 'billing_cycle', 'first_year_revenue']


## Test integrity checks

Before computing any metrics, I verify the test was set up correctly. If the
randomisation is broken or the data has anomalies, the analysis below is
meaningless.

### Sample ratio mismatch (SRM)

The 50/50 split should hold. I test this with a chi-squared goodness-of-fit
test against the expected 50/50 split. SRM is a serious failure mode if
detected, the analysis should not proceed without diagnosis.

In [22]:
#Test Integrity checks 
#Check 1: Variant assignment balance
print("Variant counts:")
print(test['variant'].value_counts())

print("\nVariant proportions:")
print(test['variant'].value_counts(normalize=True).round(4))  


Variant counts:
control      120686
treatment    120441
Name: variant, dtype: int64

Variant proportions:
control      0.5005
treatment    0.4995
Name: variant, dtype: float64


In [27]:
#Check 2: Sample Ratio Mismatch (SRM) test  
# Observed counts
observed = test['variant'].value_counts().sort_index()
print(f"Observed: {observed.tolist()}")

# Expected counts under perfect 50/50
expected = [len(test) / 2, len(test) / 2]
print(f"Expected (50/50): {expected}")

# Chi-squared test
chi2, p_value = chisquare(f_obs=observed, f_exp=expected)
print(f"\nChi-squared statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.001:
    print("  WARNING: Sample ratio mismatch detected (p < 0.001). Investigate before trusting results.")
elif p_value < 0.05:
    print("  Mild SRM signal (p < 0.05). Worth a closer look but probably acceptable.")
else:
    print(" No sample ratio mismatch. Safe to proceed.")

Observed: [120686, 120441]
Expected (50/50): [120563.5, 120563.5]

Chi-squared statistic: 0.2489
P-value: 0.6178
 No sample ratio mismatch. Safe to proceed.


### Variant balance across key dimensions

Even with overall balance, randomisation could fail within specific segments.
I check balance by company size, country, and traffic source , the three
dimensions used in pre-registered subgroup analyses.

**Result:** all segments balanced within ±1pp of 50/50. Randomisation looks clean.

In [28]:
#Check 3: Variant balance across key dimensions
# Check variant balance by company size
print("Variant share by company size:")
balance_by_size = pd.crosstab(test['company_size'], test['variant'], normalize='index').round(4)
print(balance_by_size)

# Check variant balance by country
print("\nVariant share by country:")
balance_by_country = pd.crosstab(test['country'], test['variant'], normalize='index').round(4)
print(balance_by_country)

# Check variant balance by traffic source
print("\nVariant share by traffic source:")
balance_by_source = pd.crosstab(test['traffic_source'], test['variant'], normalize='index').round(4)
print(balance_by_source)

Variant share by company size:
variant       control  treatment
company_size                    
1-4            0.5022     0.4978
20-49          0.4993     0.5007
200+           0.5020     0.4980
5-19           0.4998     0.5002
50-199         0.5008     0.4992

Variant share by country:
variant  control  treatment
country                    
AU        0.4987     0.5013
CA        0.5035     0.4965
DE        0.5029     0.4971
FR        0.4922     0.5078
NL        0.4996     0.5004
Other     0.4984     0.5016
UK        0.5039     0.4961
US        0.5004     0.4996

Variant share by traffic source:
variant         control  treatment
traffic_source                    
direct           0.5049     0.4951
email            0.5013     0.4987
organic          0.4977     0.5023
paid_search      0.5021     0.4979
paid_social      0.5000     0.5000
referral         0.5004     0.4996


In [29]:
#Check 4: Data sanity 
# Missing values
print("Missing values per column:")
print(test.isna().sum())

# Are 'plan' and 'billing_cycle' missing exactly when 'signed_up' is False?
no_signup_with_plan = test[(~test['signed_up']) & (test['plan'].notna())]
signup_with_no_plan = test[(test['signed_up']) & (test['plan'].isna())]
print(f"\nNon-signups with a plan recorded: {len(no_signup_with_plan)}")
print(f"Signups with no plan recorded: {len(signup_with_no_plan)}")

# Duplicate visit IDs?
print(f"\nDuplicate visit_ids: {test['visit_id'].duplicated().sum()}")

# Daily volumes — any suspicious gaps or spikes?
daily_volume = test.groupby('visit_date').size()
print(f"\nDaily volume: min={daily_volume.min()}, max={daily_volume.max()}, mean={daily_volume.mean():.0f}")

Missing values per column:
visit_id                   0
visit_date                 0
variant                    0
company_size               0
country                    0
traffic_source             0
signed_up                  0
plan                  230356
billing_cycle         230356
first_year_revenue         0
dtype: int64

Non-signups with a plan recorded: 0
Signups with no plan recorded: 0

Duplicate visit_ids: 0

Daily volume: min=4274, max=11144, mean=8612


**Integrity check results:**

- **Sample ratio:** 50.05% / 49.95% across 241,127 visitors. 
  Chi-squared test p-value = 0.62 — no sample ratio mismatch.
- **Segment-level balance:** every company size, country, and traffic
  source segment sits within ±1pp of 50/50. Randomisation is uniform
  across pre-registered subgroup dimensions.
- **Data sanity:** 230,356 missing values in `plan` and `billing_cycle`,
  all corresponding exactly to non-signups (every signup has a plan
  recorded; no non-signup has a plan recorded). No duplicate visit IDs.
  Daily volume range (4,274 to 11,144) is consistent with the pre-period
  weekly pattern; no tracking outages.

The test is internally valid. Proceeding to metric analysis.

# Primary Metric

In [34]:
# Filter to signups only 
signups = test[test['signed_up']].copy()

#Annual share by variant 
signups['is_annual'] = (signups['billing_cycle'] == 'annual')
annual_share = signups.groupby('variant')['is_annual'].mean().round(4) 

print("\nSignups by variant:")
print(signups['variant'].value_counts()) 

lift_pp = (annual_share['treatment'] - annual_share['control']) * 100
print(f"\nLift: {lift_pp:.2f} percentage points")


Signups by variant:
control      5688
treatment    5083
Name: variant, dtype: int64

Lift: 8.29 percentage points


In [37]:
# Counts for the test
control_signups = signups[signups['variant'] == 'control']
treatment_signups = signups[signups['variant'] == 'treatment']

control_annual = (control_signups['billing_cycle'] == 'annual').sum()
treatment_annual = (treatment_signups['billing_cycle'] == 'annual').sum()

n_control = len(control_signups)
n_treatment = len(treatment_signups)

# Two-proportion z-test
counts = np.array([control_annual, treatment_annual])
nobs = np.array([n_control, n_treatment])
z_stat, p_value = proportions_ztest(counts, nobs)

# Confidence intervals for each proportion
control_ci = proportion_confint(control_annual, n_control, alpha=0.05, method='wilson')
treatment_ci = proportion_confint(treatment_annual, n_treatment, alpha=0.05, method='wilson')

print("Primary metric — Annual share of signups")
print(f"\nControl:    {control_annual:,} / {n_control:,} = {control_annual/n_control:.4f}")
print(f"  95% CI: [{control_ci[0]:.4f}, {control_ci[1]:.4f}]")
print(f"\nTreatment:  {treatment_annual:,} / {n_treatment:,} = {treatment_annual/n_treatment:.4f}")
print(f"  95% CI: [{treatment_ci[0]:.4f}, {treatment_ci[1]:.4f}]")

print(f"\nDifference: {(treatment_annual/n_treatment - control_annual/n_control)*100:.2f} percentage points")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n Statistically significant lift in primary metric.")
else:
    print("\n No statistically significant lift detected.")

Primary metric — Annual share of signups

Control:    1,110 / 5,688 = 0.1951
  95% CI: [0.1851, 0.2057]

Treatment:  1,413 / 5,083 = 0.2780
  95% CI: [0.2658, 0.2905]

Difference: 8.28 percentage points
Z-statistic: -10.1336
P-value: 0.000000

 Statistically significant lift in primary metric.


## Primary metric result

**Annual share of signups lifted from 19.51% (control) to 27.80% (treatment).**

| Variant | Annual share | 95% CI | Signups | Annual signups |
|---|---|---|---|---|
| Control | 19.51% | [18.51%, 20.57%] | 5,688 | 1,110 |
| Treatment | 27.80% | [26.58%, 29.05%] | 5,083 | 1,413 |
| **Difference** | **+8.28pp** | (CIs do not overlap) | -605 | +303 |

**Statistical significance:** z = -10.13, p ≈ 0 (well below the 0.05 threshold).

**Practical significance:** the lift of 8.28pp is well above the 5pp MDE I designed
the test to detect, and comfortably above the 3pp shipping threshold I
pre-registered. This is a clear lift in the primary metric.

**Initial flag :** the treatment arm had ~600 fewer signups than control, despite balanced visitor allocation.
This is the first sign of the guardrail breaking and is consistent with the team's concern that the
new design might scare off some visitors.
I will quantify this in the next section.

# Checking Guadrail Metrics

In [39]:
# Guardrail 1: Overall signup conversion rate
control_conv = test[test['variant'] == 'control']['signed_up'].mean()
treatment_conv = test[test['variant'] == 'treatment']['signed_up'].mean()

# Counts for the z-test
control_total = (test['variant'] == 'control').sum()
treatment_total = (test['variant'] == 'treatment').sum()
control_signups_count = test[test['variant'] == 'control']['signed_up'].sum()
treatment_signups_count = test[test['variant'] == 'treatment']['signed_up'].sum()

# Two-proportion z-test
counts = np.array([control_signups_count, treatment_signups_count])
nobs = np.array([control_total, treatment_total])
z_stat_g1, p_value_g1 = proportions_ztest(counts, nobs)

# Confidence intervals
control_conv_ci = proportion_confint(control_signups_count, control_total, alpha=0.05, method='wilson')
treatment_conv_ci = proportion_confint(treatment_signups_count, treatment_total, alpha=0.05, method='wilson')

# Print
print("Guardrail 1 — Overall signup conversion rate")
print(f"\nControl:    {control_signups_count:,} / {control_total:,} = {control_conv:.4f}")
print(f"  95% CI: [{control_conv_ci[0]:.4f}, {control_conv_ci[1]:.4f}]")
print(f"\nTreatment:  {treatment_signups_count:,} / {treatment_total:,} = {treatment_conv:.4f}")
print(f"  95% CI: [{treatment_conv_ci[0]:.4f}, {treatment_conv_ci[1]:.4f}]")

diff_pp = (treatment_conv - control_conv) * 100
print(f"\nDifference: {diff_pp:.2f} percentage points")
print(f"Z-statistic: {z_stat_g1:.4f}")
print(f"P-value: {p_value_g1:.6f}")

# Compare against pre-registered threshold
print(f"\nPre-registered threshold: -0.3pp")
if diff_pp < -0.3 and p_value_g1 < 0.05:
    print("  Guardrail BREACHED: significant drop beyond pre-registered tolerance")
elif p_value_g1 < 0.05:
    print("  Significant change but within tolerance")
else:
    print(" No significant change in guardrail")

Guardrail 1 — Overall signup conversion rate

Control:    5,688 / 120,686 = 0.0471
  95% CI: [0.0459, 0.0483]

Treatment:  5,083 / 120,441 = 0.0422
  95% CI: [0.0411, 0.0434]

Difference: -0.49 percentage points
Z-statistic: 5.8563
P-value: 0.000000

Pre-registered threshold: -0.3pp
  Guardrail BREACHED: significant drop beyond pre-registered tolerance


## Guardrail 1 result — Overall signup conversion rate

**Treatment reduced signup conversion from 4.71% (control) to 4.22% (treatment).**

| Variant | Conversion rate | 95% CI | Visitors | Signups |
|---|---|---|---|---|
| Control | 4.71% | [4.59%, 4.83%] | 120,686 | 5,688 |
| Treatment | 4.22% | [4.11%, 4.34%] | 120,441 | 5,083 |
| **Difference** | **−0.49pp** | (CIs do not overlap) | — | −605 |

**Statistical significance:** z = 5.86, p ≈ 0 (well below the 0.05 threshold).

**Practical significance:** the drop is roughly a 10% relative decline in 
conversion (0.49 / 4.71). In absolute terms, treatment lost ~600 signups 
across the 4-week test period — around 150 per week — that control would 
otherwise have produced.

**Pre-registered threshold check:** I committed in advance to flag a guardrail 
breach if signup conversion dropped by more than 0.3pp with significance.
The observed −0.49pp drop **breaches that threshold.**

**Initial verdict on shipping criteria:** at this point, the test does NOT 
meet the pre-registered shipping bar. The primary metric passed (lift of 
+8.28pp ≫ 3pp threshold), but the guardrail failed (−0.49pp drop > 0.3pp 
tolerance). Per my Section 1 decision criteria, this scenario fits 
"**iterate**" rather than "ship" — the directional signal is real but the 
cost is too high.

**Two more checks before locking in the verdict:**

1. **First-year revenue per visitor (Guardrail 2)** - annual subscriptions 
   are billed upfront for a year, so the higher annual share might offset 
   the volume drop in revenue terms. If revenue per visitor is flat or 
   positive, the case for iterate (rather than kill) gets stronger.
2. **Pre-registered subgroup analyses** - if the conversion drop is 
   concentrated in specific segments (e.g. small companies), the right 
   action might be a segmented rollout rather than a full redesign.

In [42]:
# Revenue per visitor by variant
control_rev = test[test['variant'] == 'control']['first_year_revenue']
treatment_rev = test[test['variant'] == 'treatment']['first_year_revenue']

control_rev_mean = control_rev.mean()
treatment_rev_mean = treatment_rev.mean()
diff_rev = treatment_rev_mean - control_rev_mean

# Welch's t-test 
t_stat, p_value_rev = ttest_ind(control_rev, treatment_rev, equal_var=False)

#  a 95% CI for the difference, since revenue isn't normally distributed
np.random.seed(42)
diffs = []
for _ in range(1000):
    c_sample = control_rev.sample(len(control_rev), replace=True).mean()
    t_sample = treatment_rev.sample(len(treatment_rev), replace=True).mean()
    diffs.append(t_sample - c_sample)
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

print("Guardrail 2 — First-year revenue per visitor")
print(f"\nControl:    ${control_rev_mean:.2f}")
print(f"Treatment:  ${treatment_rev_mean:.2f}")
print(f"\nDifference: ${diff_rev:.2f} per visitor")
print(f"95% CI on the difference: [${ci_low:.2f}, ${ci_high:.2f}]")
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_value_rev:.6f}")

# Relative change
relative = (diff_rev / control_rev_mean) * 100
print(f"Relative change: {relative:.2f}%")

Guardrail 2 — First-year revenue per visitor

Control:    $7.95
Treatment:  $7.60

Difference: $-0.36 per visitor
95% CI on the difference: [$-0.71, $-0.01]

T-statistic: 2.0787
P-value: 0.037646
Relative change: -4.50%


## Guardrail 2 result — First-year revenue per visitor

**Treatment reduced revenue per visitor from $7.95 to $7.60.**

| Variant | Revenue per visitor | 
|---|---|
| Control | $7.95 |
| Treatment | $7.60 |
| **Difference** | **−$0.36** |

**95% CI on the difference (bootstrapped):** [−$0.71, −$0.01]
**Statistical significance:** t = 2.08, p = 0.038
**Relative change:** −4.5%

**Interpretation:** the higher annual share in treatment (where users pay for 
12 months upfront) does not fully compensate for the conversion drop. Revenue 
per visitor declines, though only marginally — the upper bound of the CI is 
−$0.01, meaning the effect is **just barely** statistically significant.

**Pre-registered threshold check:** I committed in advance to require revenue 
per visitor be "flat or positive" for shipping. The observed result is 
significantly negative (albeit small in magnitude). **This breaches the 
pre-registered threshold.**

**Caveat on the result:**

- The CI nearly crosses zero. The plausible range of true effects spans from 
  "noticeably worse" to "essentially identical." A repeat experiment could 
  yield p > 0.05 just from random variation. I'd treat this as "directionally 
  negative" rather than "definitively negative."
- This metric measures *first-year* revenue and assumes monthly users churn 
  on average after 6 months. The real economic comparison depends on 
  longer-horizon retention, which we can't measure within this test.

## Combined headline

Both guardrails breached. The primary lifted decisively, but at a cost.

| Metric | Result vs threshold | Verdict |
|---|---|---|
| Annual share (primary) | +8.28pp vs ≥3pp ⟶ pass | 
| Signup conversion (G1) | −0.49pp vs ≤−0.3pp tolerance ⟶ breach |  
| Revenue per visitor (G2) | −4.5% vs "flat or positive" ⟶ breach |  

Per my pre-registered decision rules from Section 1, this scenario fits 
**"iterate"** - the primary metric lifted meaningfully (≥3pp) AND the 
guardrails broke. The signal is real but the cost is too high. The version 
we tested isn't the right one.

Before locking that in, I need to check the **pre-registered subgroup 
analyses**. If the conversion drop is concentrated in specific segments 
(e.g. small companies) while other segments are unaffected, the right 
action might be a segmented rollout rather than a full iteration.

In [44]:
#Slicing it by different fields 
#By company size — hypothesis: small companies more price-sensitive, will bounce more
#By traffic source — hypothesis: lower-intent traffic more affected
#By country — no strong prior, just a "no surprises" check

## Subgroup 1: By company size

In [46]:
def subgroup_analysis(df, segment_col, segment_value):
    """Compute primary and guardrail metrics for one segment."""
    segment = df[df[segment_col] == segment_value]
    
    # Signup conversion (Guardrail 1)
    control_total = (segment['variant'] == 'control').sum()
    treatment_total = (segment['variant'] == 'treatment').sum()
    control_signups = ((segment['variant'] == 'control') & (segment['signed_up'])).sum()
    treatment_signups = ((segment['variant'] == 'treatment') & (segment['signed_up'])).sum()
    
    if control_total == 0 or treatment_total == 0:
        return None
    
    control_conv = control_signups / control_total
    treatment_conv = treatment_signups / treatment_total
    
    # Annual share (Primary) — only among signups
    seg_signups = segment[segment['signed_up']]
    control_sign = seg_signups[seg_signups['variant'] == 'control']
    treatment_sign = seg_signups[seg_signups['variant'] == 'treatment']
    
    if len(control_sign) == 0 or len(treatment_sign) == 0:
        annual_control = annual_treatment = annual_lift = None
    else:
        annual_control = (control_sign['billing_cycle'] == 'annual').mean()
        annual_treatment = (treatment_sign['billing_cycle'] == 'annual').mean()
        annual_lift = (annual_treatment - annual_control) * 100
    
    return {
        'segment': segment_value,
        'visitors_control': control_total,
        'visitors_treatment': treatment_total,
        'conv_control': control_conv,
        'conv_treatment': treatment_conv,
        'conv_diff_pp': (treatment_conv - control_conv) * 100,
        'annual_control': annual_control,
        'annual_treatment': annual_treatment,
        'annual_lift_pp': annual_lift,
        'signups_control': len(control_sign),
        'signups_treatment': len(treatment_sign),
    }


# Run for each company size

print("SUBGROUP ANALYSIS — BY COMPANY SIZE")


results = []
size_order = ['1-4', '5-19', '20-49', '50-199', '200+']
for size in size_order:
    r = subgroup_analysis(test, 'company_size', size)
    if r:
        results.append(r)

results_df = pd.DataFrame(results)
results_df = results_df[[
    'segment',
    'visitors_control', 'visitors_treatment',
    'conv_control', 'conv_treatment', 'conv_diff_pp',
    'annual_control', 'annual_treatment', 'annual_lift_pp',
]]

print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if abs(x) < 1 else f"{x:.2f}"))

SUBGROUP ANALYSIS — BY COMPANY SIZE
segment  visitors_control  visitors_treatment  conv_control  conv_treatment  conv_diff_pp  annual_control  annual_treatment  annual_lift_pp
    1-4             21629               21436        0.0375          0.0300       -0.7546          0.1047            0.1493            4.46
   5-19             38579               38617        0.0425          0.0356       -0.6956          0.1402            0.2141            7.39
  20-49             26588               26665        0.0504          0.0467       -0.3708          0.2039            0.2862            8.23
 50-199             21647               21576        0.0542          0.0530       -0.1212          0.2481            0.3351            8.70
   200+             12243               12147        0.0591          0.0560       -0.3155          0.3191            0.4176            9.86


For 1-4 companies: bad lift, big conversion damage , treatment is clearly worse  
For mid-market (20-199): strong lift, small or no conversion damage ,treatment is clearly better  
For enterprise (200+): best lift, modest conversion damage , likely net positive



In [47]:
# Revenue per visitor by company size and variant
rev_by_size = (
    test.groupby(['company_size', 'variant'])['first_year_revenue']
    .mean()
    .unstack()
    .reindex(['1-4', '5-19', '20-49', '50-199', '200+'])
)
rev_by_size['diff'] = rev_by_size['treatment'] - rev_by_size['control']
rev_by_size['relative_pct'] = (rev_by_size['diff'] / rev_by_size['control']) * 100

print("Revenue per visitor by company size:")
print(rev_by_size.round(2))

Revenue per visitor by company size:
variant       control  treatment  diff  relative_pct
company_size                                        
1-4              4.68       3.97 -0.71        -15.20
5-19             5.46       4.91 -0.55         -9.99
20-49            8.58       8.21 -0.37         -4.32
50-199          11.94      12.03  0.10          0.80
200+            13.18      13.28  0.10          0.78


For SMB (1-49 employees): treatment damages revenue. Don't ship the new design to these segments. Iterate.  
For mid-market and enterprise (50+): treatment is revenue-neutral but lifts annual share. Ship even if revenue is flat, the better mix (more annual subs) is strategically valuable for the business.

# Subgroup 2: By traffic source

In [49]:

print("SUBGROUP ANALYSIS — BY TRAFFIC SOURCE")
print("=" * 90)

source_results = []
for src in ['organic', 'direct', 'referral', 'email', 'paid_search', 'paid_social']:
    r = subgroup_analysis(test, 'traffic_source', src)
    if r:
        source_results.append(r)

source_df = pd.DataFrame(source_results)
source_df = source_df[[
    'segment',
    'visitors_control', 'visitors_treatment',
    'conv_control', 'conv_treatment', 'conv_diff_pp',
    'annual_control', 'annual_treatment', 'annual_lift_pp',
]]

print(source_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if abs(x) < 1 else f"{x:.2f}"))

# Revenue cut
rev_by_source = (
    test.groupby(['traffic_source', 'variant'])['first_year_revenue']
    .mean()
    .unstack()
)
rev_by_source['diff'] = rev_by_source['treatment'] - rev_by_source['control']
rev_by_source['relative_pct'] = (rev_by_source['diff'] / rev_by_source['control']) * 100

print("\nRevenue per visitor by traffic source:")
print(rev_by_source.round(2))

SUBGROUP ANALYSIS — BY TRAFFIC SOURCE
    segment  visitors_control  visitors_treatment  conv_control  conv_treatment  conv_diff_pp  annual_control  annual_treatment  annual_lift_pp
    organic             41948               42334        0.0483          0.0435       -0.4858          0.1889            0.2770            8.82
     direct             18357               18003        0.0528          0.0469       -0.5960          0.1979            0.2761            7.81
   referral             12033               12013        0.0577          0.0501       -0.7562          0.1931            0.2890            9.60
      email              7271                7234        0.0510          0.0445       -0.6513          0.2480            0.2857            3.77
paid_search             26732               26512        0.0422          0.0386       -0.3610          0.1949            0.2656            7.08
paid_social             14345               14345        0.0346          0.0314       -0.3207     

Paid_social is least hurt on revenue, and referral is most hurt.
This is genuinely surprising. Referral users are usually the highest-intent acquisition cohort. Two possible reasons:
- Referral users may be coming with strong expectations from a friend's recommendation about a specific monthly experience. Seeing the "annual default" disrupts the mental model.
- Could also be sample-related- referral has the smallest visitor counts in the data, so the revenue estimate is noisier.   
The −$0.98 might be partly random.  


 The variation across traffic sources is much smaller than the variation across company sizes.  
 All sources land in the −0.5% to −10% range. Compare to company size, which spans from −15% to +0.8%.   
 So company size is the dominant signal; traffic source is a secondary effect.

# Subgroup 3: By country

In [50]:
print("=" * 90)
print("SUBGROUP ANALYSIS — BY COUNTRY")
print("=" * 90)

country_results = []
for country in ['US', 'UK', 'DE', 'CA', 'AU', 'FR', 'NL', 'Other']:
    r = subgroup_analysis(test, 'country', country)
    if r:
        country_results.append(r)

country_df = pd.DataFrame(country_results)
country_df = country_df[[
    'segment',
    'visitors_control', 'visitors_treatment',
    'conv_control', 'conv_treatment', 'conv_diff_pp',
    'annual_control', 'annual_treatment', 'annual_lift_pp',
]]

print(country_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if abs(x) < 1 else f"{x:.2f}"))

# Revenue cut
rev_by_country = (
    test.groupby(['country', 'variant'])['first_year_revenue']
    .mean()
    .unstack()
)
rev_by_country['diff'] = rev_by_country['treatment'] - rev_by_country['control']
rev_by_country['relative_pct'] = (rev_by_country['diff'] / rev_by_country['control']) * 100

print("\nRevenue per visitor by country:")
print(rev_by_country.round(2))

SUBGROUP ANALYSIS — BY COUNTRY
segment  visitors_control  visitors_treatment  conv_control  conv_treatment  conv_diff_pp  annual_control  annual_treatment  annual_lift_pp
     US             48332               48248        0.0507          0.0444       -0.6295          0.2016            0.2846            8.31
     UK             15870               15622        0.0461          0.0418       -0.4262          0.1860            0.2680            8.19
     DE             12132               11994        0.0461          0.0411       -0.4973          0.2039            0.3022            9.83
     CA              9700                9564        0.0491          0.0436       -0.5471          0.1765            0.2758            9.93
     AU              8340                8382        0.0526          0.0474       -0.5274          0.2005            0.2494            4.89
     FR              7113                7338        0.0389          0.0379       -0.1058          0.1625            0.2554      

Every country except "Other" shows a revenue decline, ranging from −2.3% (DE) to −10.2% (NL).  
The treatment is consistently negative on revenue across countries, with magnitude varying. 

## Subgroup analysis - By traffic source

Hypothesis: lower-intent traffic (paid_social, paid_search) would show larger 
negative effects on signup conversion. Higher-intent traffic (organic, direct, 
referral) would be relatively protected.


**The hypothesis is partially disconfirmed.** The variation across traffic 
sources is smaller than expected, and the ordering does not match my prior:

- **Paid_social** had the *least* revenue impact (−3.5%), not the most.
- **Referral** had the *most* revenue impact (−9.9%), despite being a 
  high-intent channel.

Possible explanation: referral users may arrive with strong expectations 
shaped by a friend's recommendation of the existing experience. The annual-
default redesign may disrupt the mental model they came in with. (This is 
speculation — referral has small sample sizes, so part of the effect could 
be noise.)

Importantly, **the magnitude of variation across traffic sources is small 
compared to company size** — all sources land within a 10pp band on revenue 
change, vs. a 16pp band across company sizes. **Company size is the dominant 
segmentation signal; traffic source is a secondary effect.**

## Subgroup analysis - By country

Hypothesis: This was a "no surprises" check rather than a 
directional test.


**No country shows a directionally opposite effect.** All countries except 
"Other" show revenue declines, ranging from −2.3% (DE) to −10.2% (NL). 
The pattern is consistent with the global finding.

This passes the "no surprises" check. There's no country where the data 
suggests treatment works for them in a way that the global average masks. 
The recommendation does not need to be country-segmented.

## Putting the subgroups together

Three pre-registered cuts; one strong, one weak, one confirmatory.

| Subgroup | Strength of finding | Implication for recommendation |
|---|---|---|
| Company size | **Strong, monotonic gradient** | Segmented rollout — ship for 50+, hold for <50 |
| Traffic source | Moderate variation, hypothesis disconfirmed | Useful context, no segmentation action |
| Country | Consistent directional pattern, no outliers | No country segmentation needed |

The recommendation should anchor on **company size**. The new pricing page 
works for mid-market and enterprise customers but damages SMB acquisition.

# Section 3: Recommendation

## Recommendation: **Iterate but with a segmented launch path**

The pre-registered shipping criteria are not met (both guardrails breached), 
so I do not recommend rolling out the redesign as-is. However, the company-
size subgroup analysis reveals that the failure is concentrated in SMB 
customers (<50 employees), while the redesign performs cleanly for mid-market 
and enterprise (50+).

The right path is a two-track plan:

1. **Ship the redesign immediately to mid-market and enterprise visitors 
   (50+ employees).** The annual-share lift is strongest in these segments 
   (+8.7pp to +9.9pp), and revenue per visitor is roughly neutral 
   (+0.8%, statistically indistinguishable from zero).

2. **Iterate the design for SMB visitors (<50 employees) before any rollout 
   there.** The conversion damage in this segment is severe (15-20% 
   relative drop in revenue per visitor), and shipping as-is would 
   meaningfully harm SMB acquisition.




## The trade-off in business terms

The pricing redesign exposes a real strategic question that the test 
clarifies but doesn't resolve on its own. The team has to choose between 
three priorities:

1. **Acquisition volume** - the redesign reduces total signups. Especially 
   for SMB, this is a meaningful customer base loss.
2. **Revenue mix** - the redesign successfully shifts customers toward 
   annual contracts, which retain better and produce more predictable 
   revenue.
3. **First-year revenue per visitor** - the immediate economic outcome, 
   which is roughly flat for 50+ companies and clearly negative for <50.

For 50+ segments, all three priorities are aligned with shipping: more 
annual customers, no measurable revenue cost. For <50 segments, the 
priorities conflict — the redesign delivers more annual customers but at 
a substantial first-year revenue cost. Without the segmentation, the 
blanket finding would force the team to pick between these priorities 
globally. With the segmentation, the answer is simply "different 
treatment for different segments."

---

## Specific next steps

### 1. Implement the segmented rollout (50+ ships, <50 holds)

**Action:** Show the new annual-default pricing page only to visitors who 
identify their company as 50+ employees in the signup flow. Show the 
existing monthly-default page to visitors selecting <50.

**Why:** The data clearly supports this split. Revenue per visitor is 
neutral-to-positive for 50+ (+0.8% in both 50-199 and 200+ segments) and 
the annual lift is strongest there (+8.7pp and +9.9pp).

**Risk:** B2B SaaS visitors don't always select their company size before 
seeing pricing. The implementation needs to handle the case where the 
visitor hasn't yet identified themselves  likely default to the existing 
page until they do, then switch.

**Estimated impact (at 50+ segments):** assuming the lift holds at scale 
on real traffic, this would shift roughly 9pp of new 50+ signups to 
annual, lifting predictable revenue without measurable acquisition cost. 
The 50+ segments represent about 30% of total signups (roughly 3,000 of 
the 10,800 in the test).

### 2. Run a follow-up test specifically for SMB

**Action:** Design a new variant for SMB visitors that softens the price-
shock element. Specific candidates worth testing:

- Show the monthly equivalent price more prominently alongside the annual 
  headline ("$9.60/user/month, billed annually  that's $115/year")
- A "starts at $9.60/month" framing that doesn't require the full annual 
  commitment to be the default
- A hybrid where the monthly price is the headline but the annual savings 
  are visually amplified

**Why:** The current design is too aggressive for price-sensitive 
segments. A softer redesign might capture some of the annual lift 
without the conversion damage.


**Estimated effort:** ~6 weeks (design + 4-week test).

### 3. Add long-term retention measurement to any future iteration

**Action:** Instrument the test to track retention at 6 and 12 months, 
not just first-year revenue.

**Why:** First-year revenue undersells the strategic value of annual 
customers. Annual customers historically retain better, but we can't 
measure that within the 4-week test window. A future test should plan 
for this measurement upfront.

**Risk:** Longer measurement windows mean longer time-to-decision.

---

## What I don't know  and what would change my mind


**1. Long-term economics of the annual lift.** First-year revenue is 
flat-to-slightly-negative for 50+ segments. If annual customers retain 
substantially better than monthly customers (which is the team's prior), 
the lifetime economics could be net positive even where first-year is 
flat. **If LTV data shows annual customers retain >2x longer than monthly,** 
the case for shipping at 50+ becomes much stronger.

**2. Why referral was the most-hurt traffic source.** This was a surprise. 
The expected pattern was paid_social being most affected. **If a follow-up 
investigation finds referral users specifically come in with monthly 
expectations from referrer behaviour,** that might suggest a referral-
specific landing page rather than a sitewide change.

**3. Whether the SMB damage can be reversed with a softer design.** The 
follow-up test in step 2 will answer this. **If the softer SMB variant 
shows meaningful annual lift without conversion damage,** the long-term 
plan becomes "ship a redesigned version everywhere" rather than 
permanently maintaining two pricing pages.

**4. Implementation feasibility of company-size detection.** The 
segmented rollout assumes we can reliably identify SMB vs mid-market 
visitors before they see the pricing page. If implementation requires 
significant changes to the signup flow, the cost/benefit might shift.

---

## A note on what this test taught us beyond the immediate question

Two findings from this test that have value beyond the pricing page:

1. **Price sensitivity is a clean function of company size in this market.** 
   The monotonic gradient on conversion drop suggests company size is a 
   reliable proxy for price sensitivity in B2B SaaS. This is useful 
   context for any future pricing or packaging decisions.

2. **Annual default is a real lever, not a marketing trick.** The 8pp 
   lift in annual share is large, robust, and consistent across segments. 
   For customer bases that aren't price-sensitive, this is a viable 
   acquisition strategy beyond pricing-page redesigns.